# 📜 10.7 Spark RDD Basics

Welcome to **SECTION C: RDDS (HISTORICAL CONTEXT)**!

In the previous sections, we learned how Spark's brain (the Driver) and muscle (the Executors) work together to process partitioned data in RAM. 

Whenever we checked our partitions, we used a command like `df.rdd.getNumPartitions()`. You might have been wondering: *"What exactly is an RDD?"*

Before modern Spark DataFrames existed, Data Engineers had to write all their code using **RDDs**. While today 95% of your work will be with DataFrames, understanding RDDs is a rite of passage. It is the fundamental building block of Spark, and understanding it will make you a much stronger Data Engineer.

### 🎯 Learning Objectives
* Define what an **RDD** is by breaking down its acronym.
* Understand the key difference between an RDD (schema-less) and a DataFrame (structured).
* Learn how to create RDDs using `parallelize()`.
* Learn how to load external text files into RDDs.
* Understand how the RDD API fits into the historical context of Data Engineering.

## 1. What is an RDD?

**RDD** stands for **Resilient Distributed Dataset**.

Let's break down exactly what this means:

1. **Resilient:** It is fault-tolerant. If a Node catches fire and a partition is lost in RAM, Spark knows exactly how to recreate that lost data from scratch using the DAG blueprint. You don't lose your work.
2. **Distributed:** The data does not live on one computer. It is chopped up into partitions and spread across the memory of the cluster.
3. **Dataset:** It is a collection of records (like a list of text, numbers, or objects).

### 🚫 The Catch: RDDs have NO Schema
Think of a Spark DataFrame like an Excel spreadsheet. It has column headers (Name, Age, City) and strict data types. Spark knows exactly what is inside it.

An RDD is like a **raw, unformatted text file**. It is just a giant list of "stuff". Spark has no idea if the RDD contains numbers, words, or complex JSON strings. Because it has no schema, it is harder for Spark to optimize automatically.

<img src="../assets/Module_10/10_07_01.png" alt="A visual showing a glowing data object labeled 'RDD' breaking apart into smaller puzzle pieces (partitions), which fly onto different server racks to represent distribution, while a shield deflects a lightning bolt to represent resilience." width="75%">

In [6]:
# Let's set up our Spark environment first.
# Notice that while we use SparkSession, we will extract the "SparkContext" from it.
# RDDs are the oldest part of Spark, so they require the older SparkContext API!

from pyspark.sql import SparkSession

# 1. Create the modern SparkSession
spark = SparkSession.builder \
    .appName("RDD_Basics") \
    .master("local[*]") \
    .getOrCreate()

# 2. Extract the legacy SparkContext (sc) to work with RDDs
sc = spark.sparkContext

print("SparkSession and SparkContext are ready!")

SparkSession and SparkContext are ready!


## 2. Creating RDDs: The `parallelize()` Method

The easiest way to create an RDD is to take a standard Python list (which lives on one computer) and tell Spark to distribute it across the cluster.

We do this using `sc.parallelize()`.

Let's create a standard Python list of numbers and convert it into a Resilient Distributed Dataset.

In [7]:
# 1. A standard Python list (Lives in the Driver's memory only)
python_list = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
print(f"Type of python_list: {type(python_list)}")

# 2. Convert to an RDD (Distributes the list across the Executors)
my_first_rdd = sc.parallelize(python_list)
print(f"Type of my_first_rdd: {type(my_first_rdd)}")

# 3. Let's see how many partitions Spark chopped our list into
print(f"Number of partitions: {my_first_rdd.getNumPartitions()}")

Type of python_list: <class 'list'>
Type of my_first_rdd: <class 'pyspark.core.rdd.RDD'>
Number of partitions: 12


## 3. Viewing RDD Data (Actions)

If you just print `my_first_rdd`, you won't see the numbers. You will just see a pointer indicating that the data is scattered across the network.

Because of **Lazy Evaluation**, we must use an **Action** to force Spark to bring the data back to us.

* `collect()`: Gathers ALL data from all executors and brings it back to the Driver. *(Warning: Never use this on a Petabyte dataset, or your Driver will crash!)*
* `take(n)`: Safely grabs just the first 'n' elements.

In [8]:
# If we just print it, we get a memory address
print("Raw print:")
print(my_first_rdd)

# Using collect() to bring the distributed data back to the Driver as a Python list
print("\nUsing collect():")
collected_data = my_first_rdd.collect()
print(collected_data)

# Using take(3) to just peek at the first 3 elements safely
print("\nUsing take(3):")
peek_data = my_first_rdd.take(3)
print(peek_data)

Raw print:
ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:299

Using collect():
[10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

Using take(3):


[10, 20, 30]


## 4. Creating RDDs from External Files

In the real world, you won't parallelize small Python lists. You will read massive text files, server logs, or CSVs directly into an RDD.

We do this using `sc.textFile()`.

In [9]:
import os

# Let's generate a dummy log file on our local disk
log_content = """INFO: Server started
WARN: Memory usage at 80%
ERROR: User login failed
INFO: Data backup complete
ERROR: Database connection timeout"""

with open("server_logs.txt", "w") as f:
    f.write(log_content)

# Now, let's load this text file into an RDD
log_rdd = sc.textFile("server_logs.txt")

# Let's use an Action to count the total lines in our log file
total_lines = log_rdd.count()
print(f"The log file has {total_lines} lines.")

# Let's take a peek at the first 2 lines
print("First 2 lines:")
print(log_rdd.take(2))

The log file has 5 lines.
First 2 lines:
['INFO: Server started', 'WARN: Memory usage at 80%']


## 🧑‍💻 The Modern Data Ecosystem: Role Comparison

Why do we still talk about RDDs if DataFrames replaced them? 

| Feature | 🏛️ Traditional DBA | 🛠️ Data Engineer (You) | 🧠 Data Scientist |
| :--- | :--- | :--- | :--- |
| **Data Structure** | Structured Tables with rigid schemas. | **Uses DataFrames for structured data, but occasionally drops down to RDDs for extremely messy, unstructured text parsing.** | Exclusively uses Pandas/Spark DataFrames. Rarely touches RDDs. |
| **Language Style** | SQL. | **Functional Programming (using lambda functions on RDDs) vs Declarative (using SQL on DataFrames).** | Python/R scripting. |
| **Performance** | Database Engine handles optimization. | **If using RDDs, the DE must manually optimize the code. If using DataFrames, Spark optimizes it automatically.** | Relies on DE's architecture. |

---

### 📈 Career Progression Roadmap

1. **Junior DE:** Only knows how to use Spark DataFrames. If handed a massively unstructured log file that doesn't fit into a CSV format, they get stuck.
2. **Mid-Level DE:** Understands that a Spark DataFrame is actually just an RDD with a schema layered on top of it. Can comfortably use `sc.parallelize()` and `.collect()` to manipulate raw, unstructured data across the cluster.
3. **Senior DE:** Knows exactly when to avoid RDDs. A Senior DE knows that because RDDs lack a schema, Spark cannot optimize them efficiently. They migrate legacy RDD codebases to modern DataFrame APIs to save massive amounts of compute time.

### 🛠️ Skills Matrix
* **Core Object:** Resilient Distributed Dataset (RDD).
* **Creation API:** `sc.parallelize()` and `sc.textFile()`.
* **Basic Actions:** `.collect()`, `.take()`, `.count()`.

## 🔑 Key Takeaways

- **RDD (Resilient Distributed Dataset):** The original core data structure of Apache Spark. It is a fault-tolerant, partitioned collection of elements that can be operated on in parallel.
- **No Schema:** Unlike a table or a DataFrame, an RDD has no concept of columns or data types. It's just a raw list of items.
- **Creation:** You can create an RDD by parallelizing an existing Python list (`sc.parallelize()`) or by reading a file (`sc.textFile()`).
- **Actions vs Transformations:** Just like DataFrames, RDD operations are Lazy. You must use an Action like `.collect()` or `.take()` to execute the DAG and see the data.

## 📝 Knowledge Check Quiz

**Question 1:** What does the "Resilient" in RDD mean?
A) The data is compressed to save hard drive space.
B) The dataset is fault-tolerant; if a node crashes, Spark can rebuild the lost partition using the lineage (DAG) blueprint.
C) The dataset forces all values to be strings.
D) It means the data is stored permanently on a hard drive.

**Question 2:** Which of the following is a key difference between an RDD and a Spark DataFrame?
A) RDDs can only run on a single computer, while DataFrames are distributed.
B) DataFrames are lazy, but RDDs execute immediately.
C) A DataFrame has a strict Schema (columns and data types) allowing for automatic optimization, while an RDD is just a schema-less collection of raw objects.
D) RDDs are faster than DataFrames.

**Question 3:** Why is it dangerous to run `.collect()` on a massive Petabyte-scale RDD in production?
A) It deletes the RDD from memory.
B) It attempts to pull all Petabytes of distributed data back into the memory of the single Driver node, which will instantly cause an Out of Memory (OOM) crash.
C) It converts the RDD into an unreadable format.
D) It forces the cluster to shut down.

---

*Answers: 1: B, 2: C, 3: B*

In [10]:
# Clean up our session
spark.stop()
print("Spark session closed.")

Spark session closed.


### 🚀 What's Next?
Now that we know how to create an RDD, how do we actually transform the data inside it? 

In the next lesson, **10.8 RDD Transformations & Actions**, we will dive into Functional Programming and learn how to use `map()`, `filter()`, and `reduce()` to manipulate our raw, distributed data.